# Attention-model classification accuracy (Figure 4B)

Top-1 classification accuracy of the `cell_dino` attention model as a function of the per-class cell-count threshold, for both the **phase** and **fluorescence** classifiers. Two panels: gene knockouts (left) and protein complexes (right). The center line is the median across classes and the shaded band is the inter-quartile range (Q1–Q3); the x-axis is log-scaled so the early-bin gains (10 → 100 cells) stay visible.

Inputs are the per-class evaluation CSVs produced by the attention pipeline, curated into `../../data/figures/figure_4/` (see README). Protein-complex labels come from the EBI complex annotations.

| file | level | modality |
| --- | --- | --- |
| `figure_4b_gene_level_phase.csv` | gene KO | phase |
| `figure_4b_gene_level_fluorescence.csv` | gene KO | fluorescence |
| `figure_4b_ebi_level_phase.csv` | protein complex (EBI) | phase |
| `figure_4b_ebi_level_fluorescence.csv` | protein complex (EBI) | fluorescence |

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import (
    FixedLocator,
    FuncFormatter,
    MultipleLocator,
    NullLocator,
    ScalarFormatter,
)

# Keep text editable in Illustrator (SVG keeps <text> elements rather than
# path-tracing the glyphs; PDF uses TrueType instead of Type-3).
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42

FIGURES_DIR = Path("../../output/figure_4")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## CSV paths

Per-class attention-evaluation CSVs, curated into `../../data/figures/figure_4/` (see README).

In [ ]:
FIGURE_DATA = Path("../../data/figures/figure_4")

GENE_PHASE_CSV    = FIGURE_DATA / "figure_4b_gene_level_phase.csv"
GENE_FLUOR_CSV    = FIGURE_DATA / "figure_4b_gene_level_fluorescence.csv"
COMPLEX_PHASE_CSV = FIGURE_DATA / "figure_4b_ebi_level_phase.csv"
COMPLEX_FLUOR_CSV = FIGURE_DATA / "figure_4b_ebi_level_fluorescence.csv"

## Configuration

Figure 4B uses **top-1** accuracy with a **median + IQR** band on a **log** x-axis. Flip `BAND` to `"sem"` for a mean ± SEM band, or `XSCALE` to `"linear"` for a proportional cell-count axis.

In [ ]:
ACC_COL = "top1_acc"   # "top1_acc" or "top5_acc"
BAND = "iqr"           # "iqr" -> median + Q1–Q3; "sem" -> mean ± SEM
XSCALE = "log"         # "log" or "linear"

COLOR = {"phase": "#3A3A3A", "fluor": "#1F9B4A"}
LABEL = {"phase": "Phase classifier", "fluor": "Fluorescence classifier"}

## Load + normalize

Different eval CSVs use different identifier columns. The protein-complex CSVs carry a `label_name` (complex name); the gene CSVs carry `gene_name`. Rename whichever is present to `class_name` for downstream uniformity.

In [ ]:
def load(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "class_name" not in df.columns:
        for cand in ("label_name", "gene_name"):
            if cand in df.columns:
                df = df.rename(columns={cand: "class_name"})
                break
    need = {"n_cells", ACC_COL, "class_name"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"{path.name}: missing columns {miss}")
    return df


gene_phase    = load(GENE_PHASE_CSV)
gene_fluor    = load(GENE_FLUOR_CSV)
complex_phase = load(COMPLEX_PHASE_CSV)
complex_fluor = load(COMPLEX_FLUOR_CSV)

n_gene    = max(gene_phase["class_name"].nunique(), gene_fluor["class_name"].nunique())
n_complex = max(complex_phase["class_name"].nunique(), complex_fluor["class_name"].nunique())
print(f"gene KO: {n_gene} classes | protein complex: {n_complex} classes")

## Helpers

`center_and_band` returns the center line and band edges per `n_cells` bin; `plot_panel` draws the phase + fluor curves for one level (gene KO or complex).

In [ ]:
def center_and_band(grouped, band: str):
    """(center, lo, hi) for a pandas GroupBy.

    band="iqr" -> center=median, lo/hi = Q1/Q3
    band="sem" -> center=mean,   lo/hi = mean ∓ SEM (std / sqrt(n))
    """
    if band == "iqr":
        return grouped.median(), grouped.quantile(0.25), grouped.quantile(0.75)
    if band == "sem":
        center = grouped.mean()
        sem = grouped.sem().fillna(0.0)
        return center, center - sem, center + sem
    raise ValueError(f"unknown band mode: {band!r}")


def plot_panel(ax, phase_df, fluor_df, title, *, band, xscale):
    all_x: set[int] = set()
    for modality, df in (("phase", phase_df), ("fluor", fluor_df)):
        color = COLOR[modality]
        grouped = df.groupby("n_cells")[ACC_COL]
        center, lo_s, hi_s = center_and_band(grouped, band)
        xs = np.asarray(center.index, dtype=float)
        ys = np.asarray(center.values, dtype=float)
        all_x.update(int(n) for n in center.index)
        ax.fill_between(xs, lo_s.values, hi_s.values, color=color,
                        alpha=0.10, linewidth=0, zorder=2)
        ax.plot(xs, ys, "-o", color=color, linewidth=2.2, markersize=6,
                markeredgecolor="black", markeredgewidth=0.6, zorder=3,
                label=LABEL[modality])

    ax.set_xlabel("Cells/class", fontsize=11)
    ax.set_ylabel("Accuracy", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.tick_params(axis="both", labelsize=10)

    # Fixed 0%–100% y-axis, tick every 10%; tiny headroom so ~100% markers
    # don't clip into the frame.
    ax.set_ylim(0.0, 1.03)
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(round(v * 100))}%"))
    ax.grid(True, axis="y", alpha=0.25)

    # Explicit ticks at the actual cell-count slices. On log, set the scale
    # first (installs LogLocator) then override with our fixed ticks +
    # ScalarFormatter; otherwise matplotlib renders 10^2/10^3 and silently
    # drops some requested ticks.
    ticks = sorted(all_x)
    if xscale == "log":
        ax.set_xscale("log")
        ax.xaxis.set_major_locator(FixedLocator(ticks))
        ax.xaxis.set_minor_locator(NullLocator())
        fmt = ScalarFormatter()
        fmt.set_scientific(False)
        ax.xaxis.set_major_formatter(fmt)
        ax.set_xlim(min(ticks) * 0.7, max(ticks) * 1.3)
        ax.set_xticklabels([str(int(t)) for t in ticks])
    else:
        ax.set_xticks(ticks)
        ax.set_xlim(min(ticks) - max(ticks) * 0.03, max(ticks) + max(ticks) * 0.03)
        ax.set_xticklabels([str(int(t)) for t in ticks], rotation=35,
                            ha="right", rotation_mode="anchor")
    ax.legend(loc="lower right", fontsize=9, framealpha=0.9)

## Figure 4B

Two panels sharing the y-axis. Saves an SVG (paper) + PNG.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
plot_panel(axes[0], gene_phase, gene_fluor,
           title=f"Gene KO (n={n_gene:,})", band=BAND, xscale=XSCALE)
plot_panel(axes[1], complex_phase, complex_fluor,
           title=f"Protein complex (n={n_complex:,})", band=BAND, xscale=XSCALE)
# sharey hides the right panel's y-tick labels; re-enable so both panels
# are independently readable.
axes[1].tick_params(axis="y", labelleft=True, labelsize=10)
axes[1].set_ylabel("Accuracy", fontsize=11)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "eval_accuracy_curves.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "eval_accuracy_curves.png", dpi=240, bbox_inches="tight")
plt.show()

## Summary table

Center + spread per (level, modality, n_cells), recording both band conventions (mean/std/SEM and median/Q1/Q3) so the table stays in sync with whichever variant is plotted.

In [ ]:
rows = []
for level, df_p, df_f in [("gene", gene_phase, gene_fluor),
                          ("complex", complex_phase, complex_fluor)]:
    for modality, df in (("phase", df_p), ("fluor", df_f)):
        grouped = df.groupby("n_cells")[ACC_COL]
        for n in sorted(grouped.groups):
            vals = grouped.get_group(n).to_numpy()
            std = float(np.std(vals, ddof=1)) if vals.size > 1 else 0.0
            rows.append({
                "level": level, "modality": modality, "n_cells": int(n),
                "n_classes": int(df["class_name"].nunique()),
                "acc_mean": float(np.mean(vals)),
                "acc_std": std,
                "acc_sem": std / np.sqrt(vals.size) if vals.size else 0.0,
                "acc_median": float(np.median(vals)),
                "acc_q25": float(np.quantile(vals, 0.25)),
                "acc_q75": float(np.quantile(vals, 0.75)),
            })

summary = pd.DataFrame(rows)
summary.to_csv(FIGURES_DIR / "eval_accuracy_curves_summary.csv", index=False)
summary